# 05. PyTorch 基础与 MLP 体系

这一节开始进入深度学习。我们会用 PyTorch 建立最基础的训练闭环。

## 多层感知机（MLP）的形式化定义

            多层感知机是由多个全连接仿射变换与非线性激活函数交替堆叠而成的前馈神经网络。它通过逐层特征变换，将输入映射到更适合当前任务的表示空间中。

在函数逼近视角下，MLP 是一类通用非线性近似器；在表示学习视角下，MLP 通过深度组合逐层构造中间特征。

## 输入、输出与参数化方式

            输入为定长向量，网络参数包括每一层的权重矩阵和偏置向量。输出层根据任务类型不同而变化：

- 回归：线性输出
- 二分类：单 logit / sigmoid
- 多分类：多 logit / softmax

与卷积网络和序列网络不同，MLP 不假设局部结构或时间结构，因此最适合处理已向量化的特征输入。

## 结构分解与信息流

            一个标准 MLP 层块由三部分组成：

1. 线性层：完成特征重组合
2. 激活函数：引入非线性，使多层堆叠不再退化为单一线性映射
3. 可选正则化组件：如 Dropout、Norm

MLP 的关键不是“层数多”，而是“层间表示逐步重构”。因此在分析 MLP 时，应关注隐藏维度、激活函数与深度共同决定的表示容量。

## 优化目标与训练机制

            MLP 的训练依赖标准反向传播。前向传播产生预测，损失函数定义目标，反向传播计算梯度，优化器执行参数更新。

在这一过程中，激活函数形状、初始化尺度、学习率和正则化强度都会显著影响训练动态。对于 MLP 这类全连接网络，过拟合和梯度不稳定是两个最常见的问题。

## 计算复杂度、统计性质与工程代价

            全连接层的参数量与输入维度和隐藏维度乘积成正比，因此在高维输入场景下参数增长很快。

与 CNN 相比，MLP 不进行参数共享，因此对图像等高维结构化输入的参数效率较低；但在中小规模表格与向量任务中，MLP 仍然是非常重要的基线结构。

## 与相邻模型的差异

            与线性模型相比，MLP 通过非线性激活获得更强表达能力。
与 CNN 相比，MLP 不利用空间局部性。
与 Transformer 相比，MLP 没有显式的 token 间交互机制。
因此，MLP 更像深度学习的通用基础单元，而不是针对某类结构化数据的专用架构。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis("off")
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)

layers = {
    "输入层": (1.5, [6.5, 5.0, 3.5, 2.0]),
    "隐藏层1": (4.5, [6.8, 5.6, 4.4, 3.2, 2.0]),
    "隐藏层2": (7.5, [6.2, 4.8, 3.4, 2.0]),
    "输出层": (10.2, [5.6, 4.0, 2.4]),
}
for name, (x, ys) in layers.items():
    for y in ys:
        ax.add_patch(plt.Circle((x, y), 0.18, color="#4C78A8"))
    ax.text(x, 7.4, name, ha="center", fontsize=12)

layer_values = list(layers.values())
for i in range(len(layer_values) - 1):
    x1, ys1 = layer_values[i]
    x2, ys2 = layer_values[i + 1]
    for y1 in ys1:
        for y2 in ys2:
            ax.plot([x1, x2], [y1, y2], color="gray", alpha=0.25, lw=0.8)

ax.text(6.0, 0.8, "每层执行: 线性变换 -> 激活函数 -> 表示更新", ha="center", fontsize=12)
ax.set_title("多层感知机的前馈结构")
plt.show()

## 先建立直觉

            MLP 可以理解成“把很多线性变换和非线性激活堆起来”。

如果只有一层线性变换，模型只能学到比较简单的关系。
但如果你让模型经过多次“线性变换 + 激活函数”，它就能逐层提炼更复杂的表示。

一个很实用的直觉是：

- 前面层学的是比较粗糙的模式
- 后面层学的是更抽象、更任务相关的模式

这就是“深度”真正带来的价值。

## 架构是怎么工作的

            MLP 的每一层都做同样的事：

1. 接收上一层表示
2. 做线性变换
3. 通过激活函数引入非线性

如果没有激活函数，多层线性层叠起来仍然等价于一层线性变换，深度就没有意义。

所以：

- 线性层负责学习“怎么组合特征”
- 激活函数负责让模型能表达弯曲、复杂的关系
- Dropout 等模块负责防止模型记死训练集

## 训练时到底发生了什么

            深度学习训练的最基本闭环是：

1. 前向传播：输入经过网络得到预测
2. 计算损失：预测和真实答案差多少
3. 反向传播：自动求每个参数的梯度
4. 参数更新：优化器根据梯度调整参数

这一套流程一旦真正理解，后面 CNN、RNN、Transformer 只是“网络结构不同”，训练逻辑并没有变。

## 什么时候该用它

            MLP 适合：

- 向量化后的中小规模数据
- 作为深度学习入门模型
- 想理解 PyTorch 基本训练范式时

它不擅长显式利用空间结构和时序结构，所以在图像和序列任务上往往不是最优结构。

## 最常见的误区

            常见误区：

1. **误以为层数越多一定越强**
   更深的网络更难训练，也更容易过拟合。

2. **误以为只要调大 hidden_dim 就行**
   更大的宽度会增加参数量和过拟合风险。

3. **误以为 MLP 学不好图像是因为数据不够**
   更核心的原因是它没有利用图像的局部空间结构。

## 1. 感知机与多层感知机

单个神经元可以写成：

$$
z = w^T x + b,\qquad a = \phi(z)
$$

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


print("临时目录:", temp_root)

In [ ]:
x = torch.linspace(-2, 2, 50).unsqueeze(1)
y = 3 * x + 1

w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

lr = 0.1
losses = []
for _ in range(100):
    pred = x * w + b
    loss = ((pred - y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()
    b.grad.zero_()
    losses.append(loss.item())

print("学习到的 w:", round(w.item(), 4))
print("学习到的 b:", round(b.item(), 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(losses, color="darkorange")
axes[0].set_title("Autograd 示例的损失下降曲线")

with torch.no_grad():
    pred_line = x * w + b
axes[1].scatter(x.numpy(), y.numpy(), label="真实数据", alpha=0.7)
axes[1].plot(x.numpy(), pred_line.numpy(), color="red", label="拟合直线")
axes[1].set_title("Autograd 拟合结果")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target.astype(np.int64)

scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_tensor = torch.tensor(X_train)
y_train_tensor = torch.tensor(y_train)
X_test_tensor = torch.tensor(X_test)
y_test_tensor = torch.tensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=256, shuffle=False)

In [ ]:
class ShallowMLP(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=64, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, num_classes))

    def forward(self, x):
        return self.net(x)


class DeepMLP(nn.Module):
    def __init__(self, input_dim=64, hidden_dims=(128, 64), num_classes=10, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[1], num_classes),
        )

    def forward(self, x):
        return self.net(x)


def train_classifier(model, train_loader, test_loader, epochs=25, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    for _ in range(epochs):
        model.train()
        running_loss, running_correct, total = 0.0, 0, 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch_x.size(0)
            running_correct += (logits.argmax(dim=1) == batch_y).sum().item()
            total += batch_x.size(0)

        model.eval()
        test_loss, test_correct, test_total = 0.0, 0, 0
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                test_loss += loss.item() * batch_x.size(0)
                test_correct += (logits.argmax(dim=1) == batch_y).sum().item()
                test_total += batch_x.size(0)

        history["train_loss"].append(running_loss / total)
        history["train_acc"].append(running_correct / total)
        history["test_loss"].append(test_loss / test_total)
        history["test_acc"].append(test_correct / test_total)

    return history

In [ ]:
mlp_models = {"ShallowMLP": ShallowMLP(), "DeepMLP": DeepMLP()}
histories = {}
trained_models = {}

for name, model in mlp_models.items():
    history = train_classifier(model, train_loader, test_loader, epochs=25, lr=1e-3)
    histories[name] = history
    trained_models[name] = model
    print(name, "最终测试准确率:", round(history["test_acc"][-1], 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for name, history in histories.items():
    axes[0].plot(history["train_loss"], label=f"{name} train")
    axes[0].plot(history["test_loss"], linestyle="--", label=f"{name} test")
    axes[1].plot(history["test_acc"], label=name)
axes[0].set_title("MLP 训练 / 测试损失曲线")
axes[0].legend()
axes[1].set_title("MLP 测试准确率曲线")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
best_name = max(histories, key=lambda name: histories[name]["test_acc"][-1])
best_model = trained_models[best_name]

best_model.eval()
with torch.no_grad():
    logits = best_model(X_test_tensor)
    preds = logits.argmax(dim=1).numpy()

print("最佳模型:", best_name)
print("测试准确率:", round(accuracy_score(y_test, preds), 4))

plt.figure(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, preds, cmap="Blues")
plt.title(f"{best_name} 的混淆矩阵")
plt.show()